# 01 — Eksploracyjna analiza danych (US Accidents)

Zakres wg rubryki + protokołu ewaluacji:
1. **Rozkłady cech** • 2. **Braki danych i ich powody** • 3. **Wartości odstające** • 4. **Nowe cechy** • 5. **Zajawka tekstu** • 6. **Audyt leakage w `Description`** • na końcu W1: deduplikacja + `make_splits` (zamrożony podział)

> ⚠️ **Kontrakt predykcji:** model przewiduje w chwili zgłoszenia (`Start_Time`). `Distance(mi)`, `End_Time` (i czas trwania), `End_Lat/End_Lng` są znane dopiero PO zdarzeniu — w EDA analizujemy je wyłącznie jako **zmienne post-hoc** (ilustracja target leakage), nigdy jako cechy modelu. Lista zakazanych: `configs/config.yaml → forbidden_features`.
>
> **Target:** `Severity` mierzy wpływ zdarzenia na ruch drogowy (nie obrażenia).

> Dane: [Kaggle — sobhanmoosavi/us-accidents](https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents) → rozpakuj CSV do `dane/US_Accidents_March23.csv`.
> Pracujemy na próbce (`data.sample_rows` w `configs/config.yaml`); analizy na pełnym zbiorze — DuckDB.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from us_accidents_ml.config import load_settings
from us_accidents_ml.data import load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

settings = load_settings()
df = load_raw(settings)
print(df.shape)

In [ ]:
df.head(3).T

In [ ]:
df.dtypes.value_counts(), df.memory_usage(deep=True).sum() / 1e6

## 1. Braki danych — skala i *powody*

Hipotezy do zweryfikowania: braki pogodowe zależą od odległości do stacji meteo (`Airport_Code`, `Weather_Timestamp`);
`End_Lat/End_Lng` uzupełniane tylko dla części źródeł zgłoszeń (`Source`); braki `Precipitation(in)` ≠ brak opadu.

In [ ]:
missing = (
    df.isna().sum().to_frame("n_missing").assign(pct=lambda t: 100 * t.n_missing / len(df))
)
missing[missing.n_missing > 0].sort_values("pct", ascending=False).head(25)

## 2. Rozkład targetu `Severity` — niezbalansowanie klas

Decyzja do podjęcia w W2: wagi klas vs SMOTE vs łączenie klas skrajnych (por. notatka [[Z02 ML zasoby PMML Kaggle]]).

In [ ]:
counts = df[settings.data.target].value_counts().sort_index()
display(counts.to_frame("n").assign(pct=lambda t: (100 * t.n / len(df)).round(2)))
counts.plot(kind="bar", title="Rozkład Severity")
plt.show()

## 3. Wartości odstające

Cechy modelu: `Temperature(F)`, `Wind_Speed(mph)`, `Pressure(in)`, `Visibility(mi)`.
Zmienne **post-hoc** (`Distance(mi)`, czas trwania `End_Time − Start_Time`) analizujemy w osobnym bloku — tylko jako kontekst i ilustrację leakage'u, zgodnie z kontraktem predykcji.

In [ ]:
num_cols = ["Distance(mi)", "Temperature(F)", "Wind_Speed(mph)", "Pressure(in)", "Visibility(mi)"]
df[num_cols].describe(percentiles=[0.01, 0.5, 0.99]).T

## 4. Cechy czasowe i geograficzne (nowe cechy)

Plan: godzina/dzień tygodnia/miesiąc, pora doby (`Sunrise_Sunset`), stan/hrabstwo, gęstość wypadków w okolicy.
(Czas trwania zdarzenia — wykluczony: post-hoc.) Poniżej starter.

In [ ]:
start = pd.to_datetime(df["Start_Time"], format="mixed", errors="coerce")
features_time = pd.DataFrame(
    {
        "hour": start.dt.hour,
        "dayofweek": start.dt.dayofweek,
        "month": start.dt.month,
    }
)
features_time.hour.value_counts().sort_index().plot(kind="bar", title="Wypadki wg godziny")
plt.show()

## 5. `Description` — zajawka ścieżki tekstowej (W4)

Sprawdzamy, jak wygląda opis i czy jego długość/słownictwo różnicuje `Severity`.

In [ ]:
print(df["Description"].dropna().sample(5, random_state=42).to_list())
df.assign(desc_len=df["Description"].str.len()).boxplot(
    column="desc_len", by=settings.data.target, showfliers=False
)
plt.show()

## 6. Audyt leakage w `Description` (timebox 2–3 h)

Ile opisów zawiera **frazy skutkowe** (opisujące następstwa, nie samo zdarzenie)? To one „zdradzają" Severity.
Wynik audytu = lista fraz do maskowania w wariancie cenzurowanym (→ `config.yaml → text.effect_phrases`).

In [ ]:
effect_candidates = [
    "closed", "blocked", "block", "delay", "slow traffic", "stationary traffic",
    "queuing traffic", "shut", "lane closed", "road closed", "accident cleared",
]
desc = df["Description"].dropna().str.lower()
share = {p: round(100 * desc.str.contains(p, regex=False).mean(), 2) for p in effect_candidates}
pd.Series(share, name="% opisów z frazą").sort_values(ascending=False)
# TODO (timebox 2-3 h): top n-gramy per klasa (chi2/log-odds), finalna lista -> configs/config.yaml -> text.effect_phrases
# TODO: skala szablonowych duplikatów tekstów (desc.value_counts().head(20))

## Wnioski (uzupełniać w trakcie W1)

- [ ] Braki: które kolumny odrzucamy, które imputujemy (i czym uzasadniamy powody braków)
- [ ] Imbalance: wagi klas (decyzja protokołu: bez SMOTE przy 7,7 mln — udokumentować)
- [ ] Outliery: clipping / winsoryzacja / pozostawienie
- [ ] **Audyt leakage**: lista fraz skutkowych → `configs/config.yaml → text.effect_phrases`; % rekordów z frazami; skala szablonowych duplikatów tekstów
- [ ] **Split**: decyzja random-stratified vs czasowy (skok źródeł ~2020!) + skala deduplikacji (Start_Time ±15 min + geohash) → uruchomić `make_splits`, zapisać manifest
- [ ] Lista cech do W2 (bez `forbidden_features`) + co przenosimy do ETL (Spark)